# Big Data Pipeline: U.S. Border Crossing Analysis with Apache Spark and MongoDB

### Team Contributions

The project was completed through a structured division of responsibilities, ensuring both efficiency and consistency across all stages of the pipeline.

- **Muhamed Aniss Lotfy** was responsible for data preprocessing and initial exploration (Sections 1–3), as well as the analysis of seasonal patterns in border crossings (Section 9.2).

- **Jie Xu** focused on data transformation and Spark-based processing (Sections 4–5), and conducted the analysis of external factors, particularly the impact of COVID-19 on border crossings (Section 9.3).

- **Mu Zhao** handled database integration and querying (Sections 6–7), and performed advanced analysis on port growth and decline trends (Section 9.4).

- **Weiwei Zhang** was responsible for overall system design and integration, performance comparison and optimization (Section 8), long-term structural analysis (Section 9.1), and the implementation of bonus tasks. Additionally, the project lead ensured consistency across all sections and coordinated the final structure of the report.

## 1. Environment Setup

This section initializes the required environment, including Apache Spark, MongoDB connection, and necessary Python libraries.

In [1]:
# Initialize Spark
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BorderCrossingAnalysis") \
    .getOrCreate()

print("Spark is working!")

Spark is working!


In [2]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["border_crossing_db"]

print("MongoDB connected!")

MongoDB connected!


In [3]:
import pandas as pd
import matplotlib.pyplot as plt

print("Libraries loaded!")

Libraries loaded!


## 2. Data Loading and Initial Inspection

In this section, we load the dataset and perform initial inspection to understand its structure, size, and key attributes.

In [4]:
df = spark.read.csv("data/Border_Crossing_Entry_Data.csv", header=True, inferSchema=True)

print("Data loaded successfully!")

Data loaded successfully!


In [5]:
df.show(5)

+-----------+---------+---------+----------------+--------+-------+-----+--------+---------+--------------------+
|  Port Name|    State|Port Code|          Border|    Date|Measure|Value|Latitude|Longitude|               Point|
+-----------+---------+---------+----------------+--------+-------+-----+--------+---------+--------------------+
|    Hidalgo|    Texas|     2305|US-Mexico Border|Jan 2026|  Buses|  640|  26.095|  -98.271|POINT (-98.271092...|
|Brownsville|    Texas|     2301|US-Mexico Border|Jan 2026|  Buses|  264|  25.952|  -97.401|POINT (-97.40067 ...|
|    Warroad|Minnesota|     3423|US-Canada Border|Dec 2025|  Buses|    9|  48.999|  -95.377|POINT (-95.376555...|
|      Alcan|   Alaska|     3104|US-Canada Border|Nov 2025| Trucks|  547|  62.615| -141.001|POINT (-141.00144...|
|     Laredo|    Texas|     2304|US-Mexico Border|Jul 2025|  Buses| 2546|    27.5|  -99.507|POINT (-99.507412...|
+-----------+---------+---------+----------------+--------+-------+-----+--------+------

In [6]:
df.printSchema()

root
 |-- Port Name: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Port Code: integer (nullable = true)
 |-- Border: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Measure: string (nullable = true)
 |-- Value: integer (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Point: string (nullable = true)



In [7]:
print(df.count())

273391


In [8]:
print(len(df.columns))

10


In [9]:
for col in df.columns:
    print(col)

Port Name
State
Port Code
Border
Date
Measure
Value
Latitude
Longitude
Point


In [10]:
print("Total columns:", len(df.columns))

Total columns: 10


In [11]:
print("Column names:")
for col in df.columns:
    print(col)

Column names:
Port Name
State
Port Code
Border
Date
Measure
Value
Latitude
Longitude
Point


In [12]:
df.describe().show()

+-------+---------+----------+------------------+----------------+--------+--------------+------------------+------------------+------------------+--------------------+
|summary|Port Name|     State|         Port Code|          Border|    Date|       Measure|             Value|          Latitude|         Longitude|               Point|
+-------+---------+----------+------------------+----------------+--------+--------------+------------------+------------------+------------------+--------------------+
|  count|   273391|    273387|            273391|          273391|  273391|        273391|            273391|            273387|            273387|              273387|
|   mean|     NULL|      NULL|  2447.97364946176|            NULL|    NULL|          NULL| 41979.96001697203|43.909012114695585|-99.81721089518044|                NULL|
| stddev|     NULL|      NULL|1199.7471110625095|            NULL|    NULL|          NULL|180977.99503700226| 8.183531831499531| 18.23658657198981|        

In [13]:
from pyspark.sql.functions import col, when, count

df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df.columns
]).show()

+---------+-----+---------+------+----+-------+-----+--------+---------+-----+
|Port Name|State|Port Code|Border|Date|Measure|Value|Latitude|Longitude|Point|
+---------+-----+---------+------+----+-------+-----+--------+---------+-----+
|        0|    4|        0|     0|   0|      0|    0|       4|        4|    4|
+---------+-----+---------+------+----+-------+-----+--------+---------+-----+



## 3. Data Cleaning and Feature Engineering

This section prepares the dataset for analysis by handling missing values, selecting relevant columns, and extracting useful features such as year and month from the date field.

In [14]:
import pyspark.sql.functions as F

df_clean = spark.read.csv("data/Border_Crossing_Entry_Data.csv", header=True, inferSchema=True)

df_clean = df_clean.select(
    "Port Name",
    "State",
    "Border",
    "Date",
    "Measure",
    "Value"
).dropna()

df_clean.select("Date").show(5, False)

+--------+
|Date    |
+--------+
|Jan 2026|
|Jan 2026|
|Dec 2025|
|Nov 2025|
|Jul 2025|
+--------+
only showing top 5 rows


In [15]:
df_clean = df_clean.withColumn(
    "Date",
    F.to_date(
        F.concat(F.lit("01 "), F.col("Date")),
        "dd MMM yyyy"
    )
)

df_clean.select("Date").show(5, False)

+----------+
|Date      |
+----------+
|2026-01-01|
|2026-01-01|
|2025-12-01|
|2025-11-01|
|2025-07-01|
+----------+
only showing top 5 rows


In [16]:
df_clean = df_clean.withColumn("Year", F.year(F.col("Date")))
df_clean = df_clean.withColumn("Month", F.month(F.col("Date")))

df_clean.select("Date", "Year", "Month").show(5)

+----------+----+-----+
|      Date|Year|Month|
+----------+----+-----+
|2026-01-01|2026|    1|
|2026-01-01|2026|    1|
|2025-12-01|2025|   12|
|2025-11-01|2025|   11|
|2025-07-01|2025|    7|
+----------+----+-----+
only showing top 5 rows


In [17]:
import pyspark.sql.functions as F

df_clean = spark.read.csv("data/Border_Crossing_Entry_Data.csv", header=True, inferSchema=True)

df_clean = df_clean.dropna()

df_clean = df_clean.select(
    "Port Name",
    "State",
    "Border",
    "Date",
    "Measure",
    "Value"
)

df_clean = df_clean.withColumn(
    "Date",
    F.to_date(
        F.concat(F.lit("01 "), F.trim(F.col("Date"))),
        "dd MMM yyyy"
    )
)

df_clean = df_clean.withColumn("Year", F.year(F.col("Date")))
df_clean = df_clean.withColumn("Month", F.month(F.col("Date")))

df_clean.show(20)

+-----------+------------+----------------+----------+--------------+-----+----+-----+
|  Port Name|       State|          Border|      Date|       Measure|Value|Year|Month|
+-----------+------------+----------------+----------+--------------+-----+----+-----+
|    Hidalgo|       Texas|US-Mexico Border|2026-01-01|         Buses|  640|2026|    1|
|Brownsville|       Texas|US-Mexico Border|2026-01-01|         Buses|  264|2026|    1|
|    Warroad|   Minnesota|US-Canada Border|2025-12-01|         Buses|    9|2025|   12|
|      Alcan|      Alaska|US-Canada Border|2025-11-01|        Trucks|  547|2025|   11|
|     Laredo|       Texas|US-Mexico Border|2025-07-01|         Buses| 2546|2025|    7|
|       Roma|       Texas|US-Mexico Border|2025-07-01|         Buses|   67|2025|    7|
|     Calais|       Maine|US-Canada Border|2025-04-01|Bus Passengers|  318|2025|    4|
|    Warroad|   Minnesota|US-Canada Border|2025-04-01|        Trucks|  433|2025|    4|
|       Roma|       Texas|US-Mexico Border|

## 4. MongoDB Storage and Schema Design

The cleaned dataset is stored in MongoDB as a collection of documents. 

Each document represents a border crossing record with fields such as:
- Port Name
- Border
- State
- Date (Year, Month)
- Measure
- Value

Indexes are created on key fields (e.g., Border, Measure, Year) to improve query performance.

## 5. Exploratory Analysis with PySpark DataFrame API

This section performs initial exploratory data analysis using PySpark DataFrame API to understand overall patterns in the dataset.

### 5.1 Total Crossings by Border

This analysis compares the total number of crossings between the U.S.-Canada and U.S.-Mexico borders.

### 5.2 Total Crossings by Measure

This analysis identifies which transportation measures (e.g., vehicles, pedestrians, trucks) contribute the most to total crossings.

### 5.3 Total Crossings by Year

This analysis examines how total crossings change over time, revealing long-term trends and potential external impacts.

### 5.4 Top Ports by Total Crossings

This analysis identifies the busiest border ports based on total crossing volume.

## 6. Equivalent Analysis with Spark SQL

This section demonstrates the use of Spark SQL to perform equivalent analyses, highlighting the flexibility of Spark’s unified processing engine.

### 6.1 Total Crossings by Border

### 6.2 Total Crossings by Measure

### 6.3 Total Crossings by Year

### 6.4 Top Ports by Total Crossings

## 7. MongoDB Aggregation Queries

In this section, we perform similar aggregations using MongoDB’s aggregation framework to compare with Spark-based processing.

### 7.1 Total Crossings by Border

### 7.2 Total Crossings by Measure

### 7.3 Total Crossings by Year

### 7.4 Top Ports by Total Crossings

## 8. Performance Comparison and Optimization

This section evaluates query performance across different approaches, including PySpark DataFrame API, Spark SQL, and MongoDB aggregation.

We also demonstrate optimization techniques such as:
- Spark caching
- Query execution plan analysis
- MongoDB indexing

### 8.1 Performance Comparison

### 8.2 Spark Optimization

### 8.3 MongoDB Optimization

## 9. Advanced Meaningful Analyses

Each member is responsible for one analysis question.

For each subsection:
- Clearly answer the question
- Include at least one visualization
- Provide a short interpretation of the results
- Write the final dataset back to MongoDB

You are encouraged to explore beyond the minimum requirements if you find interesting patterns.

### 9.1 Long-term Changes in Border Crossing Structure

Question:
How have transportation patterns evolved over time at the U.S.-Canada and U.S.-Mexico borders?

### 9.2 Peak Month and Peak Volume for Top Ports

Question:
For the top ports at each border, which month has the highest crossing volume?

(Hint: identify both the peak month and the corresponding volume, and compare patterns across ports.)

### 9.3 Impact of External Events (COVID-19)

Question:
How did COVID-19 affect border crossing volumes over time?

(Hint: compare trends before, during, and after the pandemic, and look for disruptions and recovery patterns.)

### 9.4 Port Growth & Decline Analysis

Question:
Which ports have experienced the most significant growth or decline over time?

(Hint: compare two time points and consider both absolute and relative changes.)